In [ ]:
import gc
import sklearn
import numpy as np
import keras_tuner
import tensorflow as tf
import matplotlib.pyplot as plt

import os, sys
sys.path.append(os.path.dirname(os.path.dirname(os.path.abspath("Modules"))))

import Modules.ds_loader as ds_loader

X_train, y_train, X_val, y_val, X_test, y_test= ds_loader.load_tf_data()


In [ ]:
"""# 1-D convolutional ResNet model 
# https://pmc.ncbi.nlm.nih.gov/articles/PMC10128986/#sec012
class Resnet(keras_tuner.HyperModel):
    def residual_block(self, inputs, c_units, p_units, k_units):
        # C1 BLOCK
        x = tf.keras.layers.Conv1D(filters=c_units, kernel_size=k_units, strides=1, padding='same')(inputs)
        x = tf.keras.layers.ReLU()(x)
        x = tf.keras.layers.BatchNormalization()(x)
        
        x = tf.keras.layers.Conv1D(filters=c_units, kernel_size=k_units, strides=1, padding='same')(x)
        x = tf.keras.layers.ReLU()(x)
        x = tf.keras.layers.BatchNormalization()(x)
        # SC
        s = tf.keras.layers.Conv1D(filters=c_units, kernel_size=1, strides=1, padding='same')(inputs)
        x = tf.keras.layers.Add()([x, s])
        x = tf.keras.layers.ReLU()(x)
        x = tf.keras.layers.BatchNormalization()(x)
        x = tf.keras.layers.MaxPooling1D(p_units, strides=2)(x)
        return x


    def build(self, hp):
        gc.collect()
        tf.keras.backend.clear_session()
        # HYPERPARAMS
        #n_layer = 6
        k_units = 3
        p_units = 2
        n_layer = hp.Choice("n_layer",[1,2,3,4])
        c_units = hp.Choice("c_units", [32,64,128,256,512])
        d_units_0 = hp.Choice("d_units_0", [64,128,256,1024,2048])
        d_units_1 = hp.Choice('d_units_coef', [2,4])
        dropout_0 = hp.Float('dropout_0', min_value = 0.2, max_value=0.5, step=0.05)
        dropout_1 = hp.Float('dropout_1', min_value = 0.2, max_value=0.5, step=0.05)
        
        # INPUT LAYER
        inputs = tf.keras.Input(shape=(500,12))
        
        # RESIDUALS
        x = self.residual_block(inputs, c_units, p_units, k_units)
        filter_size = c_units
        for i in range(1, n_layer):
            #filter_size *= 2  
            x = self.residual_block(x, filter_size, p_units, k_units)

        # CLASSIFIER
        x = tf.keras.layers.Flatten()(x)
        x = tf.keras.layers.Dense(d_units_0, activation='relu')(x)
        x = tf.keras.layers.Dropout(dropout_0)(x)  
        x = tf.keras.layers.Dense(d_units_0 // d_units_1, activation='relu')(x)
        x = tf.keras.layers.Dropout(dropout_1)(x)  

        # OUTPUT
        outputs = tf.keras.layers.Dense(4, activation='softmax')(x)
        
        model = tf.keras.Model(inputs, outputs)
        optimizer = tf.keras.optimizers.Adam(
                        learning_rate=hp.Float('learning_rate', min_value=1e-4, max_value=1e-3),
                        weight_decay=hp.Choice('weight_decay',[1e-3,1e-4,1e-5,0.0])
                    )
        model.compile(
            optimizer=optimizer,
            loss="sparse_categorical_crossentropy",
            metrics=["accuracy"]
        )
        
        
        return model
    
    def fit(self, hp, model, *args, **kwargs):
        return model.fit(
            batch_size= hp.Choice("batch_size", [32]),
            *args,
            **kwargs,
        ) """

In [ ]:
"""class Resnet(keras_tuner.HyperModel):
    def residual_block(self, inputs, c_units, p_units, k_units):
        # C1 BLOCK
        x = tf.keras.layers.Conv1D(filters=c_units, kernel_size=6, strides=1, padding='same')(inputs)
        x = tf.keras.layers.ReLU()(x)
        x = tf.keras.layers.BatchNormalization()(x)
        x = tf.keras.layers.MaxPooling1D(6, 6, padding="same")(x)
        x = tf.keras.layers.Conv1D(filters=c_units, kernel_size=k_units, strides=1, padding='same')(x)
        x = tf.keras.layers.ReLU()(x)
        x = tf.keras.layers.BatchNormalization()(x)
        x = tf.keras.layers.MaxPooling1D(p_units, p_units, padding="same")(x)
        x = tf.keras.layers.Conv1D(filters=c_units, kernel_size=k_units, strides=1, padding='same')(x)
        x = tf.keras.layers.ReLU()(x)
        x = tf.keras.layers.BatchNormalization()(x)
        x = tf.keras.layers.MaxPooling1D(p_units, p_units, padding="same")(x)
    
        return x


    def build(self, hp):
        gc.collect()
        tf.keras.backend.clear_session()
        # HYPERPARAMS
        n_layer = 1
        k_units = 3
        p_units = 3

        c_units = hp.Choice("c_units", [32,64,128,256])
        d_units_0 = hp.Choice("d_units_0", [64,128,256,512,1024])
        d_units_1 = hp.Choice('d_units_coef', [2,4])
        dropout_0 = hp.Float('dropout_0', min_value = 0.2, max_value=0.5, step=0.05)
        dropout_1 = hp.Float('dropout_1', min_value = 0.2, max_value=0.5, step=0.05)
        
        # INPUT LAYER
        inputs = tf.keras.Input(shape=(constants.FINAL_SIZE,12))
        
        # RESIDUALS
        x = self.residual_block(inputs, c_units, p_units, k_units)
        filter_size = c_units
        for i in range(1, n_layer):
            #filter_size *= 2  
            x = self.residual_block(x, filter_size, p_units, k_units)

        # CLASSIFIER
        x = tf.keras.layers.Flatten()(x)
        x = tf.keras.layers.Dense(d_units_0, activation='relu')(x)
        x = tf.keras.layers.Dropout(dropout_0)(x)  
        x = tf.keras.layers.Dense(d_units_0 // d_units_1, activation='relu')(x)
        x = tf.keras.layers.Dropout(dropout_1)(x)  

        # OUTPUT
        outputs = tf.keras.layers.Dense(4, activation='softmax')(x)
        
        model = tf.keras.Model(inputs, outputs)
        optimizer = tf.keras.optimizers.Adam(
                        learning_rate=hp.Float('learning_rate', min_value=1e-5, max_value=1e-3),
                        weight_decay=hp.Choice('weight_decay',[1e-3,1e-4,1e-5,0.0])
                    )
        model.compile(
            optimizer=optimizer,
            loss="sparse_categorical_crossentropy",
            metrics=["accuracy"]
        )
        
        
        return model
    
    def fit(self, hp, model, *args, **kwargs):
        return model.fit(
            batch_size= hp.Choice("batch_size", [32]),
            *args,
            **kwargs,
        ) """

In [ ]:
class Resnet(keras_tuner.HyperModel):
    @staticmethod
    def residual_block(x, filters, kernel_size=15, dilation_rate=1, name_prefix='res'):
        shortcut = x
        
        # Main path with dilated convolutions
        x = tf.keras.layers.BatchNormalization(name=f'{name_prefix}_bn1')(x)
        x = tf.keras.layers.Activation('relu', name=f'{name_prefix}_act1')(x)
        x = tf.keras.layers.Conv1D(
            filters=filters,
            kernel_size=kernel_size,
            padding='same',
            dilation_rate=dilation_rate,
            kernel_regularizer=tf.keras.regularizers.l2(1e-4),
            name=f'{name_prefix}_conv1'
        )(x)
        
        x = tf.keras.layers.BatchNormalization(name=f'{name_prefix}_bn2')(x)
        x = tf.keras.layers.Activation('relu', name=f'{name_prefix}_act2')(x)
        x = tf.keras.layers.Conv1D(
            filters=filters,
            kernel_size=kernel_size,
            padding='same',
            kernel_regularizer=tf.keras.regularizers.l2(1e-4),
            name=f'{name_prefix}_conv2'
        )(x)
        
        # If dimensions don't match, adjust the shortcut
        if shortcut.shape[-1] != filters:
            shortcut = tf.keras.layers.Conv1D(
                filters=filters,
                kernel_size=1,
                padding='same',
                name=f'{name_prefix}_shortcut'
            )(shortcut)
        
        # Add the shortcut to the main path
        x = tf.keras.layers.add([x, shortcut], name=f'{name_prefix}_add')
        return x
    
    @staticmethod
    def attention_block(x):
        attention = tf.keras.layers.Conv1D(1, kernel_size=7, padding='same', activation='tanh')(x)
        attention = tf.keras.layers.Softmax(axis=1)(attention)
        
        # Apply attention weights to the input
        return tf.keras.layers.multiply([x, attention])
    
    @staticmethod
    def squeeze_excite_block(x, filters, ratio=16):
        se = tf.keras.layers.GlobalAveragePooling1D()(x)
        se = tf.keras.layers.Dense(filters // ratio, activation='relu')(se)
        se = tf.keras.layers.Dense(filters, activation='sigmoid')(se)
        se = tf.keras.layers.Reshape((1, filters))(se)
        return tf.keras.layers.multiply([x, se])
    
    def build(self, hp):
        input_length, input_channels = 1000,12
        num_classes = len(np.unique(y_train))
        filters_start = 32
        k_units_0 = hp.Int("k_units_0",  1, 11, 2)
        k_units_1 = hp.Int("k_units_1",  1, 11, 2)
        k_units_2 = hp.Int("k_units_2",  1, 11, 2)
        k_units_3 = hp.Int("k_units_3",  1, 11, 2)
        # Input
        inputs = tf.keras.layers.Input(shape=(input_length, input_channels), name='ecg_signal')
        
        # Initial convolution
        x = tf.keras.layers.Conv1D(
        filters=filters_start,
        kernel_size=k_units_0,
        padding='same',
        kernel_regularizer=tf.keras.regularizers.l2(1e-4),
        name='initial_conv'
        )(inputs)
        
        # First block with spatial attention
        x = self.residual_block(x, filters_start, kernel_size=k_units_1, dilation_rate=1, name_prefix='res1')
        x = self.squeeze_excite_block(x, filters_start)
        x = tf.keras.layers.MaxPooling1D(pool_size=2, name='pool1')(x)
        
        # Stack of residual blocks with increasing dilation rates
        x = self.residual_block(x, filters_start*2, kernel_size=k_units_1, dilation_rate=2, name_prefix='res2')
        x = self.squeeze_excite_block(x, filters_start*2)
        x = tf.keras.layers.MaxPooling1D(pool_size=2, name='pool2')(x)
        
        x = self.residual_block(x, filters_start*4, kernel_size=k_units_2, dilation_rate=4, name_prefix='res3')
        x = self.squeeze_excite_block(x, filters_start*4)
        x = tf.keras.layers.MaxPooling1D(pool_size=2, name='pool3')(x)
        
        x = self.residual_block(x, filters_start*8, kernel_size=k_units_3, dilation_rate=8, name_prefix='res4')
        x = self.squeeze_excite_block(x, filters_start*8)
        x = tf.keras.layers.MaxPooling1D(pool_size=2, name='pool4')(x)
        x = self.attention_block(x)
        
        # Global pooling and classification head
        x = tf.keras.layers.GlobalAveragePooling1D(name='global_avg_pool')(x)
        
        # Dropout for regularization
        x = tf.keras.layers.Dropout(0.5, name='dropout')(x)
        
        # Classification head
        x = tf.keras.layers.Dense(256, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(1e-4), name='dense1')(x)
        x = tf.keras.layers.BatchNormalization(name='bn_final')(x)
        outputs = tf.keras.layers.Dense(num_classes, activation='softmax', name='predictions')(x)
        
        # Create and compile model
        model = tf.keras.Model(inputs=inputs, outputs=outputs, name='ECG_Classifier')
        
        # Compile with Adam optimizer and appropriate loss 
        model.compile(
            optimizer=tf.keras.optimizers.Adam(
            learning_rate=hp.Float('learning_rate', 1e-5, 1e-3)),
            loss='sparse_categorical_crossentropy',
            metrics=['accuracy']
        )
        
        return model


In [ ]:
"""import tensorflow as tf
from tensorflow.keras import layers, Model, regularizers
from tensorflow.keras.optimizers import Adam

def build_ecg_classifier(input_length=1000, num_classes=4, input_channels=12, filters_start=32):
    # Input layer
    inputs = layers.Input(shape=(input_length, input_channels), name='ecg_signal')

    
    # Residual block function
    def residual_block(x, filters, kernel_size=15, dilation_rate=1, name_prefix='res'):
        # Shortcut connection
        shortcut = x
        
        # Main path with dilated convolutions
        x = layers.BatchNormalization(name=f'{name_prefix}_bn1')(x)
        x = layers.Activation('relu', name=f'{name_prefix}_act1')(x)
        x = layers.Conv1D(
            filters=filters,
            kernel_size=kernel_size,
            padding='same',
            dilation_rate=dilation_rate,
            kernel_regularizer=regularizers.l2(1e-4),
            name=f'{name_prefix}_conv1'
        )(x)
        
        x = layers.BatchNormalization(name=f'{name_prefix}_bn2')(x)
        x = layers.Activation('relu', name=f'{name_prefix}_act2')(x)
        x = layers.Conv1D(
            filters=filters,
            kernel_size=kernel_size,
            padding='same',
            kernel_regularizer=regularizers.l2(1e-4),
            name=f'{name_prefix}_conv2'
        )(x)
        
        # If dimensions don't match, adjust the shortcut
        if shortcut.shape[-1] != filters:
            shortcut = layers.Conv1D(
                filters=filters,
                kernel_size=1,
                padding='same',
                name=f'{name_prefix}_shortcut'
            )(shortcut)
        
        # Add the shortcut to the main path
        x = layers.add([x, shortcut], name=f'{name_prefix}_add')
        return x
    
    # Initial convolution
    x = layers.Conv1D(
        filters=filters_start,
        kernel_size=5,
        padding='same',
        kernel_regularizer=regularizers.l2(1e-4),
        name='initial_conv'
    )(inputs)
    
    # First block with spatial attention
    x = residual_block(x, filters_start, kernel_size=15, dilation_rate=1, name_prefix='res1')
    
    # Squeeze and Excitation block
    def squeeze_excite_block(x, filters, ratio=16):
        se = layers.GlobalAveragePooling1D()(x)
        se = layers.Dense(filters // ratio, activation='relu')(se)
        se = layers.Dense(filters, activation='sigmoid')(se)
        se = layers.Reshape((1, filters))(se)
        return layers.multiply([x, se])
    
    x = squeeze_excite_block(x, filters_start)
    x = layers.MaxPooling1D(pool_size=2, name='pool1')(x)
    
    # Stack of residual blocks with increasing dilation rates
    x = residual_block(x, filters_start*2, kernel_size=3, dilation_rate=2, name_prefix='res2')
    x = squeeze_excite_block(x, filters_start*2)
    x = layers.MaxPooling1D(pool_size=2, name='pool2')(x)
    
    x = residual_block(x, filters_start*4, kernel_size=3, dilation_rate=4, name_prefix='res3')
    x = squeeze_excite_block(x, filters_start*4)
    x = layers.MaxPooling1D(pool_size=2, name='pool3')(x)
    
    x = residual_block(x, filters_start*8, kernel_size=1, dilation_rate=8, name_prefix='res4')
    x = squeeze_excite_block(x, filters_start*8)
    x = layers.MaxPooling1D(pool_size=2, name='pool4')(x)
    
    # Add attention mechanism
    def attention_block(x):
        # Compute attention weights
        attention = layers.Conv1D(1, kernel_size=7, padding='same', activation='tanh')(x)
        attention = layers.Softmax(axis=1)(attention)
        
        # Apply attention weights to the input
        return layers.multiply([x, attention])
    
    x = attention_block(x)
    
    # Global pooling and classification head
    x = layers.GlobalAveragePooling1D(name='global_avg_pool')(x)
    
    # Dropout for regularization
    x = layers.Dropout(0.5, name='dropout')(x)
    
    # Classification head
    x = layers.Dense(256, activation='relu', kernel_regularizer=regularizers.l2(1e-4), name='dense1')(x)
    x = layers.BatchNormalization(name='bn_final')(x)
    outputs = layers.Dense(num_classes, activation='softmax', name='predictions')(x)
    
    # Create and compile model
    model = Model(inputs=inputs, outputs=outputs, name='ECG_Classifier')
    
    # Compile with Adam optimizer and appropriate loss 
    model.compile(
        optimizer=Adam(learning_rate=0.0001),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    
    return model

# Example usage
model = build_ecg_classifier(input_length=1000, num_classes=4)
model.summary()"""

In [ ]:
RDIR="../src/Results/RES_1000_00/" 
MDIR= RDIR + "RES_1000_00.keras"
CDIR= RDIR + "C_RES_1000_00.keras"
CVDIR = RDIR + "RES_1000_00_CV.keras"

tuner = keras_tuner.BayesianOptimization(
    Resnet(),
    objective='val_accuracy',
    max_trials=10,
    overwrite=False,
    directory=RDIR,
    project_name="RES_1000_00",
)
tuner.search_space_summary()


In [ ]:
"""def train_model(model, X_train, y_train, X_val, y_val, epochs=50, batch_size=32):
    # Learning rate scheduler
    lr_scheduler = tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=1e-6,
        verbose=1
    )
    
    # Early stopping
    early_stopping = tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True,
        verbose=1
    )
    
    # Model checkpoint
    model_checkpoint = tf.keras.callbacks.ModelCheckpoint(
        'best_ecg_model.h5',
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    )
    
    # Train the model
    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=epochs,
        batch_size=batch_size,
        callbacks=[lr_scheduler, early_stopping, model_checkpoint]
    )
    
    return history, model

train_model(model, X_train, y_train, X_val, y_val, epochs=100, batch_size=32)"""

In [ ]:
lr_scheduler = tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=1e-6,
        verbose=1
    )
    
early_stopping = tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True,
        verbose=1
    )
    
model_checkpoint = tf.keras.callbacks.ModelCheckpoint(
        filepath=CDIR,
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    )
tuner.search(
    X_train, y_train, 
    epochs = 150,
    validation_data=(X_val, y_val),
    callbacks=[lr_scheduler,early_stopping,model_checkpoint]
)

In [ ]:
tuner.results_summary()

In [ ]:
best_model = tuner.get_best_models(num_models=1)[0]
best_model.summary()
best_model.save(MDIR) 

In [ ]:
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0] 
print(best_hps.values)

In [ ]:
test_loss, test_accuracy = best_model.evaluate(X_test, y_test, batch_size=32)
print(f"Test Loss: {test_loss}")
print(f"Test Accuracy: {test_accuracy}")

In [ ]:
y_pred = best_model.predict(X_test)

if y_pred.shape[1] == 1:  
    y_pred_binary = (y_pred > 0.5).astype(int)
    auc = sklearn.metrics.roc_auc_score(y_test, y_pred)  
else:
    y_pred_binary = np.argmax(y_pred, axis=1)  
    auc = sklearn.metrics.roc_auc_score(y_test, y_pred, multi_class='ovr')

print("Classification Report (Test Data):")
print(sklearn.metrics.classification_report(y_test, y_pred_binary))
print(f"AUC: {auc}")

y_train_pred = best_model.predict(X_train)
y_train_pred = np.argmax(y_train_pred, axis=1)

print("Classification Report (Train Data):")
print(sklearn.metrics.classification_report(y_train, y_train_pred))

In [ ]:
import seaborn as sns
y_pred_class = np.argmax(y_pred, axis=1)  
cm = sklearn.metrics.confusion_matrix(y_test, y_pred_class, normalize='true')

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt=".2f", cmap="Blues", xticklabels=[0, 1, 2, 3], yticklabels=[0, 1, 2, 3])
plt.xlabel('Predicted Labels')
plt.ylabel('True Labels')
plt.title('Confusion Matrix')
plt.show()

In [ ]:
"""kfold = sklearn.model_selection.KFold(n_splits=10, shuffle=True, random_state=42)
fold_accuracies = []
fold_histories = []

best_accuracy = 0.0
best_model = None  

for fold, (train_idx, val_idx) in enumerate(kfold.split(X_train, y_train)):
    print(f"\n--- Fold {fold+1} ---")

    fold_callbacks = [
        tf.keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=5, restore_best_weights=True)
    ]
    X_tr, X_val_fold = X_train[train_idx], X_train[val_idx]
    y_tr, y_val_fold = y_train[train_idx], y_train[val_idx]

    model = Resnet().build(best_hps)

    history = model.fit(
        X_tr, y_tr,
        validation_data=(X_val_fold, y_val_fold),
        epochs=100,
        callbacks=fold_callbacks,
        verbose=1
    )

    val_loss, val_accuracy = model.evaluate(X_val_fold, y_val_fold, verbose=0)
    print(f"Fold {fold+1} Validation Accuracy: {val_accuracy:.4f}")
    fold_accuracies.append(val_accuracy)
    fold_histories.append(history)

    if val_accuracy > best_accuracy:
        best_accuracy = val_accuracy
        best_model = model
        model.save(CVDIR) 
        print(f"Saved best model from Fold {fold+1} with Accuracy: {val_accuracy:.4f}")
"""

In [ ]:
"""print("Cross-validation accuracies:", fold_accuracies)
print("Average CV accuracy:", np.mean(fold_accuracies))
print("Max CV accuracy:", np.max(fold_accuracies))"""

In [ ]:
cv_model = tf.keras.models.load_model(CVDIR)
cv_model.evaluate(X_test, y_test)

In [ ]:
test_loss, test_accuracy = cv_model.evaluate(X_test, y_test, batch_size=32)
print(f"Test Loss: {test_loss}")
print(f"Test Accuracy: {test_accuracy}")

In [ ]:
y_pred = cv_model.predict(X_test)

if y_pred.shape[1] == 1:  
    y_pred_binary = (y_pred > 0.5).astype(int)
    auc = sklearn.metrics.roc_auc_score(y_test, y_pred)  
else:
    y_pred_binary = np.argmax(y_pred, axis=1)  
    auc = sklearn.metrics.roc_auc_score(y_test, y_pred, multi_class='ovr')

print("Classification Report (Test Data):")
print(sklearn.metrics.classification_report(y_test, y_pred_binary))
print(f"AUC: {auc}")

y_train_pred = cv_model.predict(X_train)
y_train_pred = np.argmax(y_train_pred, axis=1)

print("Classification Report (Train Data):")
print(sklearn.metrics.classification_report(y_train, y_train_pred))

In [ ]:
y_pred_probs = cv_model.predict(X_test)
y_pred = np.argmax(y_pred_probs, axis=1)

print("Classification Report (Test Data):")
print(sklearn.metrics.classification_report(y_test, y_pred))

auc = sklearn.metrics.roc_auc_score(y_test, y_pred_probs, multi_class='ovr')
print(f"AUC (Test): {auc:.4f}")

y_train_probs = cv_model.predict(X_train)
y_train_pred = np.argmax(y_train_probs, axis=1)

print("Classification Report (Train Data):")
print(sklearn.metrics.classification_report(y_train, y_train_pred))

In [ ]:
import seaborn as sns

cm = sklearn.metrics.confusion_matrix(y_test, y_pred, normalize='true')

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt=".2f", cmap="Blues", xticklabels=[0, 1, 2, 3], yticklabels=[0, 1, 2, 3])
plt.xlabel('Predicted Labels')
plt.ylabel('True Labels')
plt.title('Confusion Matrix')
plt.show()

In [ ]:

fig, ax = plt.subplots(1, 2, figsize=(14, 6))

ax[0].plot(history.history['accuracy'], label='accuracy')
ax[0].plot(history.history['val_accuracy'], label='val_accuracy')
ax[0].set_title('Accuracy vs Val Accuracy')
ax[0].set_xlabel('Epochs')
ax[0].set_ylabel('Accuracy')
ax[0].legend()

ax[1].plot(history.history['loss'], label='loss')
ax[1].plot(history.history['val_loss'], label='val_loss')
ax[1].set_title('Loss vs Val Loss')
ax[1].set_xlabel('Epochs')
ax[1].set_ylabel('Loss')
ax[1].legend()

plt.tight_layout()
plt.show()